# WineRadar EDA (Exploratory Data Analysis)

수집된 와인 뉴스 데이터를 탐색하고 분석합니다.

**데이터 개요:**
- 596개 URL
- 6개 소스
- 기간: 2025-09-29 ~ 2025-11-19
- 평균 점수: 1.33

In [ ]:
# 필요한 라이브러리 임포트
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
from datetime import datetime, timezone
from collections import Counter

# 한글 폰트 설정 (Windows)
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# 스타일 설정
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# 데이터베이스 연결
db_path = Path('../data/wineradar.duckdb')
conn = duckdb.connect(str(db_path), read_only=True)

## 1. 데이터 로드 및 기본 정보

In [ ]:
# URLs 테이블 로드
df_urls = conn.execute('SELECT * FROM urls').df()
df_entities = conn.execute('SELECT * FROM url_entities').df()

print(f"URLs 데이터: {len(df_urls):,}개")
print(f"Entities 데이터: {len(df_entities):,}개")
print(f"\n컬럼 목록 ({len(df_urls.columns)}개):")
for col in df_urls.columns:
    print(f"  - {col}")

In [ ]:
# 기본 통계
df_urls.describe()

In [ ]:
# 데이터 타입 확인
df_urls.dtypes

## 2. 소스별 분석

In [ ]:
# 소스별 기사 수
source_counts = df_urls['source_name'].value_counts()
print("=== 소스별 기사 수 ===")
print(source_counts)

# 시각화
fig, ax = plt.subplots(figsize=(10, 6))
source_counts.plot(kind='barh', ax=ax, color='purple')
ax.set_xlabel('기사 수')
ax.set_ylabel('소스')
ax.set_title('소스별 수집 기사 수')
plt.tight_layout()
plt.show()

In [ ]:
# 소스별 평균 점수
source_scores = df_urls.groupby('source_name')['score'].agg(['mean', 'median', 'std', 'count'])
source_scores = source_scores.sort_values('mean', ascending=False)
print("=== 소스별 점수 통계 ===")
print(source_scores)

# 시각화
fig, ax = plt.subplots(figsize=(10, 6))
source_scores['mean'].plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('평균 점수')
ax.set_ylabel('소스')
ax.set_title('소스별 평균 점수')
plt.tight_layout()
plt.show()

## 3. 시간별 분석

In [ ]:
# 날짜 컬럼 변환
df_urls['published_date'] = pd.to_datetime(df_urls['published_at']).dt.date
df_urls['published_month'] = pd.to_datetime(df_urls['published_at']).dt.to_period('M')

# 일별 기사 수
daily_counts = df_urls['published_date'].value_counts().sort_index()
print("=== 최근 10일 기사 수 ===")
print(daily_counts.tail(10))

# 시각화
fig, ax = plt.subplots(figsize=(14, 6))
daily_counts.plot(ax=ax, marker='o', linewidth=2, markersize=4)
ax.set_xlabel('날짜')
ax.set_ylabel('기사 수')
ax.set_title('일별 기사 수집 추이')
ax.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# 월별 통계
monthly_stats = df_urls.groupby('published_month').agg({
    'url': 'count',
    'score': 'mean',
    'source_name': 'nunique'
}).rename(columns={'url': 'count', 'score': 'avg_score', 'source_name': 'unique_sources'})

print("=== 월별 통계 ===")
print(monthly_stats)

## 4. 지역/대륙별 분석

In [ ]:
# 대륙별 분포
continent_counts = df_urls['continent'].value_counts()
print("=== 대륙별 기사 수 ===")
print(continent_counts)

# 시각화
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# 막대 그래프
continent_counts.plot(kind='bar', ax=ax1, color='coral')
ax1.set_xlabel('대륙')
ax1.set_ylabel('기사 수')
ax1.set_title('대륙별 기사 수')
ax1.tick_params(axis='x', rotation=45)

# 파이 차트
continent_counts.plot(kind='pie', ax=ax2, autopct='%1.1f%%', startangle=90)
ax2.set_ylabel('')
ax2.set_title('대륙별 비율')

plt.tight_layout()
plt.show()

In [ ]:
# 국가별 분포
country_counts = df_urls['country'].value_counts().head(15)
print("=== Top 15 국가별 기사 수 ===")
print(country_counts)

# 시각화
fig, ax = plt.subplots(figsize=(10, 8))
country_counts.plot(kind='barh', ax=ax, color='teal')
ax.set_xlabel('기사 수')
ax.set_ylabel('국가')
ax.set_title('Top 15 국가별 기사 수')
plt.tight_layout()
plt.show()

## 5. 신뢰도 & 목적별 분석

In [ ]:
# 신뢰도 계층별 분포
trust_counts = df_urls['trust_tier'].value_counts()
print("=== 신뢰도 계층별 기사 수 ===")
print(trust_counts)

# 시각화
fig, ax = plt.subplots(figsize=(10, 6))
trust_counts.plot(kind='bar', ax=ax, color='darkgreen')
ax.set_xlabel('신뢰도 계층')
ax.set_ylabel('기사 수')
ax.set_title('신뢰도 계층별 기사 분포')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# 정보 목적별 분석 (JSON 필드)
def extract_info_purposes(row):
    if pd.isna(row['info_purpose']):
        return []
    try:
        if isinstance(row['info_purpose'], str):
            return json.loads(row['info_purpose'])
        return row['info_purpose']
    except:
        return []

# 모든 목적 추출
all_purposes = []
for idx, row in df_urls.iterrows():
    purposes = extract_info_purposes(row)
    all_purposes.extend(purposes)

purpose_counts = pd.Series(all_purposes).value_counts()
print("=== 정보 목적별 빈도 ===")
print(purpose_counts)

# 시각화
fig, ax = plt.subplots(figsize=(10, 6))
purpose_counts.plot(kind='barh', ax=ax, color='mediumpurple')
ax.set_xlabel('기사 수')
ax.set_ylabel('정보 목적')
ax.set_title('정보 목적별 기사 분포')
plt.tight_layout()
plt.show()

## 6. 엔티티 분석

In [ ]:
# 엔티티 타입별 분포
entity_type_counts = df_entities['entity_type'].value_counts()
print("=== 엔티티 타입별 개수 ===")
print(entity_type_counts)

# 시각화
fig, ax = plt.subplots(figsize=(10, 6))
entity_type_counts.plot(kind='bar', ax=ax, color='indianred')
ax.set_xlabel('엔티티 타입')
ax.set_ylabel('개수')
ax.set_title('엔티티 타입별 분포')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# 가장 많이 언급된 포도 품종
grape_entities = df_entities[df_entities['entity_type'] == 'grape_variety']
top_grapes = grape_entities['entity_value'].value_counts().head(20)
print("=== Top 20 포도 품종 ===")
print(top_grapes)

# 시각화
fig, ax = plt.subplots(figsize=(10, 8))
top_grapes.plot(kind='barh', ax=ax, color='rebeccapurple')
ax.set_xlabel('언급 횟수')
ax.set_ylabel('포도 품종')
ax.set_title('가장 많이 언급된 포도 품종 (Top 20)')
plt.tight_layout()
plt.show()

In [ ]:
# 가장 많이 언급된 와인 지역
region_entities = df_entities[df_entities['entity_type'] == 'region']
top_regions = region_entities['entity_value'].value_counts().head(20)
print("=== Top 20 와인 지역 ===")
print(top_regions)

# 시각화
fig, ax = plt.subplots(figsize=(10, 8))
top_regions.plot(kind='barh', ax=ax, color='darkorange')
ax.set_xlabel('언급 횟수')
ax.set_ylabel('와인 지역')
ax.set_title('가장 많이 언급된 와인 지역 (Top 20)')
plt.tight_layout()
plt.show()

In [ ]:
# 가장 많이 언급된 와이너리
winery_entities = df_entities[df_entities['entity_type'] == 'winery']
top_wineries = winery_entities['entity_value'].value_counts().head(20)
print("=== Top 20 와이너리 ===")
print(top_wineries)

# 시각화
fig, ax = plt.subplots(figsize=(10, 8))
top_wineries.plot(kind='barh', ax=ax, color='darkred')
ax.set_xlabel('언급 횟수')
ax.set_ylabel('와이너리')
ax.set_title('가장 많이 언급된 와이너리 (Top 20)')
plt.tight_layout()
plt.show()

## 7. 점수 분석

In [ ]:
# 점수 분포
print("=== 점수 통계 ===")
print(df_urls['score'].describe())

# 시각화
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 히스토그램
df_urls['score'].hist(bins=30, ax=ax1, color='skyblue', edgecolor='black')
ax1.set_xlabel('점수')
ax1.set_ylabel('빈도')
ax1.set_title('점수 분포 (히스토그램)')
ax1.axvline(df_urls['score'].mean(), color='red', linestyle='--', label='평균')
ax1.legend()

# 박스플롯
df_urls.boxplot(column='score', ax=ax2)
ax2.set_ylabel('점수')
ax2.set_title('점수 분포 (박스플롯)')

plt.tight_layout()
plt.show()

In [ ]:
# 점수 구간별 기사 수
score_bins = [0, 1, 2, 3, 4, 5]
score_labels = ['0-1', '1-2', '2-3', '3-4', '4-5']
df_urls['score_range'] = pd.cut(df_urls['score'], bins=score_bins, labels=score_labels)
score_range_counts = df_urls['score_range'].value_counts().sort_index()

print("=== 점수 구간별 기사 수 ===")
print(score_range_counts)

# 시각화
fig, ax = plt.subplots(figsize=(10, 6))
score_range_counts.plot(kind='bar', ax=ax, color='gold')
ax.set_xlabel('점수 구간')
ax.set_ylabel('기사 수')
ax.set_title('점수 구간별 기사 분포')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

## 8. 상관관계 분석

In [ ]:
# URL당 엔티티 수 계산
entities_per_url = df_entities.groupby('url').size().reset_index(name='entity_count')
df_merged = df_urls.merge(entities_per_url, on='url', how='left')
df_merged['entity_count'] = df_merged['entity_count'].fillna(0)

# 점수와 엔티티 수의 관계
correlation = df_merged[['score', 'entity_count']].corr()
print("=== 점수와 엔티티 수의 상관관계 ===")
print(correlation)

# 시각화
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(df_merged['entity_count'], df_merged['score'], alpha=0.5, s=20)
ax.set_xlabel('엔티티 수')
ax.set_ylabel('점수')
ax.set_title(f'점수 vs 엔티티 수 (상관계수: {correlation.iloc[0, 1]:.3f})')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. 요약 및 인사이트

In [ ]:
print("=" * 60)
print("WineRadar 데이터 분석 요약")
print("=" * 60)
print(f"\n📊 전체 통계:")
print(f"  - 총 기사 수: {len(df_urls):,}개")
print(f"  - 활성 소스: {df_urls['source_name'].nunique()}개")
print(f"  - 수집 기간: {df_urls['published_at'].min().date()} ~ {df_urls['published_at'].max().date()}")
print(f"  - 평균 점수: {df_urls['score'].mean():.2f}")

print(f"\n🌍 지역 분포:")
print(f"  - 가장 많은 대륙: {continent_counts.index[0]} ({continent_counts.iloc[0]}개)")
print(f"  - 가장 많은 국가: {country_counts.index[0]} ({country_counts.iloc[0]}개)")

print(f"\n📰 소스 분석:")
print(f"  - 최다 기사 소스: {source_counts.index[0]} ({source_counts.iloc[0]}개)")
print(f"  - 최고 평점 소스: {source_scores.index[0]} ({source_scores.iloc[0]['mean']:.2f}점)")

print(f"\n🏷️ 엔티티 통계:")
print(f"  - 총 엔티티: {len(df_entities):,}개")
print(f"  - 엔티티 타입: {df_entities['entity_type'].nunique()}개")
if len(top_grapes) > 0:
    print(f"  - 최다 언급 포도: {top_grapes.index[0]} ({top_grapes.iloc[0]}회)")
if len(top_regions) > 0:
    print(f"  - 최다 언급 지역: {top_regions.index[0]} ({top_regions.iloc[0]}회)")

print(f"\n📈 일일 평균:")
days_active = (df_urls['published_at'].max() - df_urls['published_at'].min()).days + 1
print(f"  - 일평균 기사 수: {len(df_urls) / days_active:.1f}개/일")

print("\n" + "=" * 60)

In [ ]:
# 연결 종료
conn.close()
print("✅ 분석 완료!")